In [ ]:
from lnb_hakatons import PROJECT_DIR
import pandas as pd


DATA_DIR = "data/cleaned"

CLEAN_PATH = PROJECT_DIR / DATA_DIR / "recenzijas_clean_check.csv"
data_df = pd.read_csv(CLEAN_PATH)

CLEAN_PATH = PROJECT_DIR / DATA_DIR / "recenzijas_autori.csv"
autori_df = pd.read_csv(CLEAN_PATH)

CLEAN_PATH = PROJECT_DIR / DATA_DIR / "recenzijas_recenzenti.csv"
recenzenti_df = pd.read_csv(CLEAN_PATH)

CLEAN_PATH = PROJECT_DIR / DATA_DIR / "recenzijas_recenzetie_darbi.csv"
recenzetie_darbi_df = pd.read_csv(CLEAN_PATH)

CLEAN_PATH = PROJECT_DIR / DATA_DIR / "recenzijas_institucijas.csv"
institucijas_df = pd.read_csv(CLEAN_PATH)


FILTERED_OUT_PATH = PROJECT_DIR / DATA_DIR / "recenzijas_filtered_out.csv"
filtered_out_df = pd.read_csv(FILTERED_OUT_PATH).reset_index(drop=True)

DATA_DIR = PROJECT_DIR / "data/Mākslu kritika"
DATA_FILE = "cleaned-records-33-wide.csv"
original_data_df = pd.read_csv(DATA_DIR / DATA_FILE, sep=";")


# Data stats

In [ ]:
len(original_data_df)

In [ ]:
len(data_df)

In [ ]:
11298+85

## Recenziju autori

In [ ]:
len(data_df[data_df["Recenzenti"].isna()])

In [ ]:
len(original_data_df)

In [ ]:
original_data_df["PRIEKŠMETS - ŽANRS (655)"].value_counts().head(30)

In [ ]:
no_genre = original_data_df["PRIEKŠMETS - ŽANRS (655)"].isna() & original_data_df["PRIEKŠMETS - ŽANRS - 2 (655)"].isna()
len(original_data_df[no_genre])

In [ ]:
len(original_data_df[no_genre]) / len(original_data_df)

In [ ]:
original_data_df[no_genre].iloc[-2]["RAKSTA NOSAUKUMS (245)"]

### Authors

In [ ]:
original_data_df

In [ ]:
isna = (
    data_df["AUTORS (100)_a"].isna() | (data_df["AUTORS (100)_4"] == "aqt")
) & (
    data_df["PAPILDRAKSTS (700)_a"].isna()]
 data_df["PAPILDRAKSTS (700)_a"].isna() & data_df["PAPILDRAKSTS - 2 (700)_a"].isna()
print(len(data_df[isna]))
len(data_df[~isna]) / len(data_df)

### Authors

## Works

In [ ]:
original_data_df["AVOTA NOSAUKUMS (773)"].value_counts().head(30)

In [ ]:
autori_df[autori_df.Autors.str.contains("..")]

In [ ]:
autori_df

In [ ]:
# startified sample, 5 items from each Recenzijas tips
seed = 42
samples = []
for tips in data_df["Recenzijas tips"].unique():
    samples.append(data_df.query("`Recenzijas tips` == @tips").sample(5))
samples = pd.concat(samples, ignore_index=True)
len(samples)

In [ ]:
samples.to_csv(PROJECT_DIR / DATA_DIR / "recenzijas_sample.csv", index=False)

In [ ]:
pd.set_option('display.max_colwidth', None)
col = "PRIEKŠMETS - INSTITŪCIJA - 3 (610)"
col2 = "PRIEKŠMETS - INSTITŪCIJA - 4 (610)"
col3 = "RAKSTA NOSAUKUMS (245)"
original_data_df[~original_data_df[col2].isna()][[col3, col, col2]]

In [ ]:
data_df[data_df.Autori.isna()]

In [ ]:
from ast import literal_eval

(
    data_df
    .assign(**{
        "Recenzenti un gadi (list)": lambda x: x["Recenzenti un gadi (list)"].apply(literal_eval),
        "Autori un gadi (list)": lambda x: x["Autori un gadi (list)"].apply(literal_eval),
    })
    .assign(lens = lambda x: x["Autori un gadi (list)"].apply(lambda x: len(x) if isinstance(x, list) else 0))
    .query("lens == 0")
)[["Galvenais autors", "Autori un gadi (list)"]]


In [ ]:
data_df[data_df.Autori.isna()]

In [ ]:
# 
(
    original_data_df[["RECENZĒTĀ FILMA VAI IZRĀDE (630)"]]
    .copy()
    .assign(lens=lambda x: x["RECENZĒTĀ FILMA VAI IZRĀDE (630)"].str.len())
    .sort_values(by="lens", ascending=False)
).iloc[10]["RECENZĒTĀ FILMA VAI IZRĀDE (630)"]


In [ ]:
list(original_data_df.columns)

In [ ]:
data_df

In [ ]:
list(data_df.columns)

In [ ]:
data_df[["visi_recenzenti", "visi_autori_gadi"]]

In [ ]:
# list(original_data_df.columns)

In [ ]:
data_df

In [ ]:
pd.set_option('display.max_colwidth', 500)
filtered_out_df[filtered_out_df["RECENZĒTĀ FILMA VAI IZRĀDE (630)_a"].isna()==False][[
    "AUTORS (100)_a",
    "RECENZĒTĀ FILMA VAI IZRĀDE (630)_a",
    "RAKSTA NOSAUKUMS (245)_a",
    "RAKSTA NOSAUKUMS (245)_b",
]]

In [ ]:
# original_data_df["PRIEKŠMETS - ŽANRS - 2 (655)"].value_counts()

In [ ]:
data_df.head(1)

In [ ]:
recenziju_tipu_skaits = (
    data_df
    .groupby("Recenzijas tips")
    .agg(skaits=("Recenzijas tips", "count"))
    .reset_index()
)
recenziju_tipu_skaits.sort_values(by="skaits", ascending=False)

In [ ]:
tipi = [
    "Teātra recenzijas",
    "Literatūras recenzijas",
    "Kinofilmu recenzijas",
    "Mūzikas recenzijas",
    "Izstāžu recenzijas",
    "Operas recenzijas",
]
time_series = []
for tips in tipi:
    time_series.append(
        data_df
        .query("`Recenzijas tips` == @tips")
        .groupby("Gads")
        .agg(
            skaits=("Recenzijas tips", "count"),
        )
        .reset_index()
        .rename(columns={"Gads": "gads"})
        .assign(recenzijas_tips=tips)
)

time_series = pd.concat(time_series, ignore_index=True)

In [ ]:
import altair as alt

# Bar chart showing review type counts
bar_chart = alt.Chart(recenziju_tipu_skaits).mark_bar().add_selection(
    alt.selection_interval()
).encode(
    y=alt.Y('Recenzijas tips:N', 
            sort='-x'),  # Sort by count descending
    x=alt.X('skaits:Q'),
    # color=alt.Color('recenzijas_tips:N', 
                    # legend=None),
    tooltip=['Recenzijas tips:N', 'skaits:Q']
).properties(
    width=600,
    height=400,
).interactive()

bar_chart


In [ ]:
# Line chart showing time series trends for each review type
line_chart = alt.Chart(time_series.query("gads >= 2015")).mark_line(
    strokeWidth=3,
    point=True
).encode(
    x=alt.X(
        'gads:O', 
        # title='gads',
        scale=alt.Scale(zero=False)
    ),
    y=alt.Y('skaits:Q', 
            scale=alt.Scale(zero=False)),
    color=alt.Color('recenzijas_tips:N'),
    # strokeDash=alt.StrokeDash('recenzijas_tips:N'),
    tooltip=['gads:O', 'skaits:Q', 'recenzijas_tips:N']
).properties(
    width=700,
    height=500,
).interactive()

line_chart


In [ ]:
# Combined visualization: Bar chart and line chart side by side
combined_chart = alt.hconcat(
    bar_chart,
    line_chart,
    spacing=50
).resolve_scale(
    color='independent'
).properties(
    title='Review Analysis: Distribution and Trends'
)

combined_chart


In [ ]:
# Alternative: Stacked area chart for better trend comparison
area_chart = alt.Chart(time_series).mark_area(
    opacity=0.7,
    stroke='white',
    strokeWidth=1
).encode(
    x=alt.X('gads:O', 
            title='Year',
            scale=alt.Scale(zero=False)),
    y=alt.Y('skaits:Q', 
            title='Number of Reviews',
            stack='normalize'),  # Normalize to show proportions
    color=alt.Color('recenzijas_tips:N', 
                    title='Review Type',
                    scale=alt.Scale(scheme='category20')),
    tooltip=['gads:O', 'skaits:Q', 'recenzijas_tips:N']
).properties(
    width=700,
    height=400,
    title='Proportional Review Trends Over Time'
).interactive()

area_chart


# Check filtered out data

In [ ]:
filtered_out_df["PRIEKŠMETS - ŽANRS (655)_a"].isna().sum()

In [ ]:
_col = "PRIEKŠMETS - ŽANRS (655)_a"
filtered_out_df[_col].value_counts().head(20)

In [ ]:
# filtered_out_df[filtered_out_df[_col] == "Izstādes."].iloc[1500]

In [ ]:
filtered_out_df["PRIEKŠMETS - TEMATS (650)_a"].value_counts().head(50)

In [ ]:
filtered_out_df["PRIEKŠMETS - TEMATS (650)_a"].isna().sum()

In [ ]:
pd.set_option('display.max_colwidth', None)
cols = [
    "AUTORS (100)_4",
    "RAKSTA NOSAUKUMS (245)_a", 
    "RAKSTA NOSAUKUMS (245)_b",
    "PRIEKŠMETS - ŽANRS (655)_a",
    "PRIEKŠMETS - ŽANRS (655)_x",
    "PRIEKŠMETS - TEMATS (650)_a",
    "filter_reason",
]
filtered_out_df[filtered_out_df["PRIEKŠMETS - TEMATS (650)_a"] == "Teātra iestudējumi"][cols]

In [ ]:
# Option 1: Set pandas display options to show all content
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)  # Show full content of each column
pd.set_option('display.width', None)  # Don't wrap output
pd.set_option('display.max_rows', None)  # Show all rows (if needed)

# Now try displaying the row again
filtered_out_df.iloc[1591]


In [ ]:
_col = "RECENZĒTĀ FILMA VAI IZRĀDE (630)_g"
len(filtered_out_df[~filtered_out_df[_col].isna()])

In [ ]:
filtered_out_df[~filtered_out_df[_col].isna()][_col].value_counts()

In [ ]:
filtered_out_df[filtered_out_df[_col] == "(teātra izrāde : Regnārs Vaivars)."].iloc[0]